# CVE2Patch — Raccolta e ranking preliminare

**Fase di discovery — versione 3.** Obiettivo: un primo pool di CVE Linux con contesto pubblico utile alla futura valutazione lato host. Le evidenze raccolte non sono test di riproducibilità o di detection.

### Fondamento metodologico
| Riferimento | Scelta adottata nel notebook |
|---|---|
| [CVEfixes — Bhandari et al., 2021](https://doi.org/10.1145/3475960.3475985) | Acquisizione automatizzata NVD e associazione di dati esterni ai record CVE. |
| [The Anatomy of a Vulnerability Database, 2023](https://doi.org/10.1016/j.jss.2023.111679) | Integrazione di informazioni da più database, qui mediante CVE ID. |
| [Jacobs et al., 2020](https://doi.org/10.1093/cybsec/tyaa015) | Distinzione tra fonti di exploit con copertura diversa. Non adottiamo il modello di previsione del rischio. |
| [Mu et al., 2018](https://www.usenix.org/conference/usenixsecurity18/presentation/mu) | Dare rilievo alla disponibilità delle informazioni utili alla riproduzione; la presenza di documentazione non ne prova il successo. |
| [CVE-GENIE — Ullah et al., 2025; v2 2026](https://arxiv.org/abs/2509.01835v2) | Raccogliere risorse ed evidenze per preparare il contesto prima della futura fase agentica; nessuna ricostruzione o generazione in questa fase. |

**Scelte specifiche CVE2Patch (riunione, tesi5.txt):** periodo dal 2019, Linux, AV Network/Local, kernel separato, match deterministici su fonti prestabilite, ranking a pesi e revisione manuale. I paper motivano il metodo, non validano i pesi. Il punteggio è un indice documentale, non CVSS, EPSS, probabilità o efficacia di Falco.

Eseguire dalla cartella originale del notebook. I file in `CVE2Patch/data/raw/` vengono riutilizzati; gli output derivati vengono rigenerati. Nessun exploit viene scaricato o eseguito. I feed annuali NVD sono per anno dell'identificativo: il filtro sulla pubblicazione vale per i record acquisiti, ma non recupera eventuali ID anteriori al 2019 pubblicati dal 2019 in poi.


## 1. Librerie

Dipendenze per Python 3.11+ (requests, pandas, PyYAML).

In [ ]:
# CVE2Patch - CVE Selection
# Cella 1: Importazione delle librerie

import requests
import pandas as pd
import json
import re
import tomllib
import yaml
import hashlib

from pathlib import Path
from datetime import datetime, timezone

print("Librerie importate correttamente.")
import gzip
from IPython.display import display
from urllib.parse import urlparse


## 2. Configurazione

Percorsi e cache invariati. I cinque pesi sono contributi binari per fonte, provvisori (non ricavati numericamente dai paper). La preferenza per Local e il segnale host testuale sono criteri secondari, non bonus.


In [ ]:
BASE_DIR = Path.cwd() / "CVE2Patch"
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "output"
for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

START_DATE = "2019-01-01"
END_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
ATTACK_VECTORS = ["NETWORK", "LOCAL"]
POOL_SIZE = 100  # valore del notebook originale; modificabile

# Pesi esplorativi: non misurano severità né successo della detection.
WEIGHTS = {
    "has_vulhub": 30,
    "msf_compatible": 25,
    "edb_compatible": 15,
    "has_github": 5,
    "falco_syscall": 25,
}
assert all(isinstance(w, int) and w >= 0 for w in WEIGHTS.values())
assert sum(WEIGHTS.values()) == 100, "I pesi devono sommare a 100."
assert POOL_SIZE > 0
print(f"Periodo: {START_DATE} - {END_DATE}")
print(f"Directory: {BASE_DIR}")


## 3. NVD — Acquisizione dei feed

Download dei soli file annuali mancanti; i file compressi preesistenti vengono riutilizzati.

In [ ]:
NVD_DIR = RAW_DIR / "nvd_feeds"
NVD_DIR.mkdir(parents=True, exist_ok=True)

NVD_BASE_URL = "https://nvd.nist.gov/feeds/json/cve/2.0"

NVD_FILES = []

for year in range(int(START_DATE[:4]), int(END_DATE[:4]) + 1):

    filename = f"nvdcve-2.0-{year}.json.gz"
    file_path = NVD_DIR / filename

    if not file_path.exists():

        print(f"Download dataset {year}...")

        url = f"{NVD_BASE_URL}/{filename}"
        temp_path = file_path.with_suffix(".part")

        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()

            with temp_path.open("wb") as f:
                for chunk in response.iter_content(1024 * 1024):
                    f.write(chunk)

        temp_path.replace(file_path)

    else:
        print(f"Dataset {year} già disponibile.")

    NVD_FILES.append(file_path)

print("\nAcquisizione completata.")
print(f"Dataset annuali disponibili: {len(NVD_FILES)}")


## 4. NVD — Estrazione e filtri di base

Il filtro usa CPE strutturati e distingue software vulnerabile (`vulnerable: true`) da requisiti ambientali. Le semplici menzioni testuali non bastano. I casi non risolti sono conservati; il kernel rimane sempre separato. Un CPE non è una prova sperimentale.

In [ ]:
def extract_cvss(cve):
    for version in ["cvssMetricV40", "cvssMetricV31", "cvssMetricV30", "cvssMetricV2"]:
        entries = cve.get("metrics", {}).get(version, [])
        if entries:
            metric = next((m for m in entries if m.get("type") == "Primary"), entries[0])
            data = metric.get("cvssData", {})
            return data.get("baseScore"), data.get("attackVector", data.get("accessVector")), data.get("version")
    return None, None, None

LINUX_PRODUCTS = {"linux_kernel", "linux", "ubuntu_linux", "ubuntu", "debian_linux", "debian",
                  "fedora", "centos", "rocky_linux", "almalinux", "amazon_linux", "amazon_linux_2",
                  "amazon_linux_2023", "arch_linux", "opensuse", "leap", "tumbleweed", "oracle_linux",
                  "raspberry_pi_os", "kali_linux", "red_hat_enterprise_linux"}
NONLINUX = re.compile(r"^(windows(?:_|$)|macos(?:_|$)|mac_os(?:_|$)|os_x(?:_|$)|ios(?:_|$)|ipados(?:_|$)|android(?:_|$)|freebsd(?:_|$)|openbsd(?:_|$)|netbsd(?:_|$)|solaris(?:_|$)|aix(?:_|$))")
LINUX_TEXT = re.compile(r"\b(?:linux|ubuntu|debian|fedora|centos|red hat|suse|almalinux|rocky linux)\b", re.I)
KERNEL_TEXT = re.compile(r"(?:^\s*(?:in the |the )?linux kernel\b|\b(?:flaw|vulnerability|bug|issue|race condition|use-after-free)\b[^.]{0,65}\b(?:in|within) (?:the )?linux kernel\b)", re.I)


def parse_cpe(match):
    criteria = match.get("criteria", "")
    parts = re.split(r"(?<!\\):", criteria.lower())
    if len(parts) != 13 or parts[:2] != ["cpe", "2.3"]:
        return None
    part, vendor, product, target_sw = parts[2], parts[3], parts[4], parts[10]
    linux = (part == "o" and (product in LINUX_PRODUCTS or product.startswith(("enterprise_linux", "linux_enterprise", "ubuntu_linux", "debian_linux")))) or target_sw == "linux"
    other = (part == "o" and bool(NONLINUX.match(product))) or bool(NONLINUX.match(target_sw))
    return {"criteria": criteria, "vulnerable": match.get("vulnerable") is True,
            "linux": linux, "other": other and not linux, "part": part,
            "kernel": part == "o" and vendor == "linux" and product == "linux_kernel"}


def walk_cpe(obj):
    # Trascuriamo rami negati. Le configurazioni complesse rimangono euristiche.
    if not isinstance(obj, dict) or obj.get("negate", False):
        return [], False
    terms = [p for m in obj.get("cpeMatch", []) if (p := parse_cpe(m)) is not None]
    context = False
    for child in obj.get("nodes", []) + obj.get("children", []):
        child_terms, child_context = walk_cpe(child)
        terms.extend(child_terms)
        context |= child_context
    # Un requisito ambientale Linux conta solo in una congiunzione con software vulnerabile.
    if str(obj.get("operator", "OR")).upper() == "AND":
        context |= (any(t["linux"] and not t["vulnerable"] for t in terms)
                    and any(t["vulnerable"] and t["part"] == "a" and not t["other"] for t in terms))
    return terms, context


def platform_evidence(cve, description):
    terms, linux_context = [], False
    for configuration in cve.get("configurations", []):
        group, context = walk_cpe(configuration)
        terms.extend(group)
        linux_context |= context
    affected = [t for t in terms if t["vulnerable"]]
    linux_direct = any(t["linux"] for t in affected)
    other_only = bool(affected) and all(t["other"] for t in affected) and not linux_context
    kernel = (any(t["kernel"] for t in affected)
              or cve.get("sourceIdentifier", "").lower() == "cve@kernel.org"
              or bool(KERNEL_TEXT.search(description)))
    linux_text = bool(LINUX_TEXT.search(description))
    if kernel:
        status = "kernel"
    elif linux_direct:
        status = "linux_direct"
    elif linux_context:
        status = "linux_context"
    elif other_only:
        status = "nonlinux_cpe"
    else:
        status = "unknown"
    return {"platform_nvd": status, "linux_direct": linux_direct, "linux_context": linux_context,
            "linux_text": linux_text, "kernel": kernel,
            "cpe": json.dumps(sorted({t["criteria"] for t in terms})),
            "vulnerable_cpe": json.dumps(sorted({t["criteria"] for t in affected}))}


records = []
for file_path in NVD_FILES:
    with gzip.open(file_path, "rt", encoding="utf-8") as f:
        data = json.load(f)
    for item in data["vulnerabilities"]:
        cve = item["cve"]
        description = next((d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"), "")
        cvss, attack_vector, cvss_version = extract_cvss(cve)
        cwes = sorted({d["value"] for w in cve.get("weaknesses", []) for d in w.get("description", [])})
        references = sorted({r["url"] for r in cve.get("references", []) if r.get("url")})
        records.append({"cve_id": cve["id"], "published": cve.get("published"),
                        "description": description, "status": cve.get("vulnStatus"),
                        "cvss": cvss, "cvss_version": cvss_version, "attack_vector": attack_vector,
                        "cwe": json.dumps(cwes), "references": json.dumps(references),
                        **platform_evidence(cve, description)})
    print(f"Letto {file_path.name}")
    del data

df_all = pd.DataFrame(records).drop_duplicates("cve_id").reset_index(drop=True)
del records
in_period = df_all["published"].fillna("").str[:10].between(START_DATE, END_DATE)
df_valid = df_all[in_period & df_all["status"].ne("Rejected")].copy()
# Kernel conservato anche se AV mancante o diverso da quelli selezionati.
df_kernel = df_valid[df_valid["kernel"]].copy()
df_non_kernel = df_valid[~df_valid["kernel"]].copy()
df_av = df_non_kernel[df_non_kernel["attack_vector"].isin(ATTACK_VECTORS)].copy()
df_candidates = df_av[df_av["platform_nvd"].isin(["linux_direct", "linux_context"])].copy()
df_unknown = df_av[df_av["platform_nvd"].eq("unknown")].copy()
df_nonlinux = df_av[df_av["platform_nvd"].eq("nonlinux_cpe")].copy()

for name, frame in {"all_cves": df_all, "kernel_cves": df_kernel,
                    "candidates": df_candidates, "platform_unknown": df_unknown,
                    "platform_nonlinux": df_nonlinux,
                    "attack_vector_excluded": df_non_kernel[~df_non_kernel["attack_vector"].isin(ATTACK_VECTORS)]}.items():
    frame.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)

print(f"Record NVD unici: {len(df_all)}")
print(f"Non rigettate nel periodo: {len(df_valid)}")
print(f"Kernel separato (tutti gli AV): {len(df_kernel)}")
print(f"Non kernel Network/Local: {len(df_av)}")
print(df_av["platform_nvd"].value_counts())
assert len(df_av) == len(df_candidates) + len(df_unknown) + len(df_nonlinux)
display(df_candidates.head())


## 5. Vulhub — Ambienti vulnerabili

Indice degli ambienti disponibili; una corrispondenza non equivale a un laboratorio testato.

In [ ]:
# Cella 5: Acquisizione dati Vulhub

VULHUB_URL = (
    "https://raw.githubusercontent.com/"
    "vulhub/vulhub/master/environments.toml"
)

vulhub_file = RAW_DIR / "vulhub_environments.toml"

# Download con caching locale
if not vulhub_file.exists():
    response = requests.get(VULHUB_URL, timeout=60)
    response.raise_for_status()
    vulhub_file.write_bytes(response.content)

# Lettura del file TOML
with vulhub_file.open("rb") as f:
    vulhub_data = tomllib.load(f)

vulhub_records = []

for env in vulhub_data["environment"]:
    for cve_id in env.get("cve", []):

        vulhub_records.append({
            "cve_id": cve_id.upper(),
            "vulhub_name": env.get("name"),
            "vulhub_app": env.get("app"),
            "vulhub_path": env.get("path"),
            "vulhub_tags": json.dumps(env.get("tags", [])),
            "vulhub_url": (
                "https://github.com/vulhub/vulhub/tree/master/"
                + env.get("path", "")
            )
        })

df_vulhub = pd.DataFrame(vulhub_records)

df_vulhub.to_csv(
    PROCESSED_DIR / "vulhub.csv", index=False
)

# Corrispondenze con le CVE candidate
vulhub_matches = df_candidates[
    df_candidates["cve_id"].isin(df_vulhub["cve_id"])
]

# Possibili CVE Linux sfuggite al primo filtro
vulhub_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_vulhub["cve_id"])
]

print(f"Ambienti Vulhub: {len(vulhub_data['environment'])}")
print(f"CVE uniche Vulhub: {df_vulhub['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(vulhub_matches)}")
print(f"Possibili CVE da recuperare: {len(vulhub_recovery)}")

display(df_vulhub.head())


## 6. Metasploit — Metadati

Acquisizione dei metadati pubblici dei moduli; si distinguono i moduli di tipo exploit.

In [ ]:
# Cella 6: Acquisizione dati Metasploit

MSF_URL = (
    "https://raw.githubusercontent.com/rapid7/"
    "metasploit-framework/master/db/modules_metadata_base.json"
)

msf_file = RAW_DIR / "metasploit_metadata.json"

# Download con caching locale
if not msf_file.exists():
    response = requests.get(MSF_URL, timeout=120)
    response.raise_for_status()
    msf_file.write_bytes(response.content)

msf_data = json.loads(
    msf_file.read_text(encoding="utf-8")
)

msf_records = []

for module in msf_data.values():

    references = module.get("references", [])

    cve_ids = set(re.findall(
        r"CVE-\d{4}-\d{4,}",
        json.dumps(references).upper()
    ))

    for cve_id in sorted(cve_ids):

        msf_records.append({
            "cve_id": cve_id,
            "msf_name": module.get("name"),
            "msf_module": module.get("fullname"),
            "msf_type": module.get("type"),
            "msf_platform": module.get("platform"),
            "msf_rank": module.get("rank"),
            "msf_description": module.get("description")
        })

df_msf = pd.DataFrame(msf_records).drop_duplicates()

df_msf.to_csv(
    PROCESSED_DIR / "metasploit.csv", index=False
)

# Moduli classificati come exploit
df_msf_exploits = df_msf[
    df_msf["msf_type"] == "exploit"
].copy()

# Incrocio con le candidate
msf_matches = df_candidates[
    df_candidates["cve_id"].isin(df_msf_exploits["cve_id"])
]

# Possibili CVE da recuperare
msf_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_msf_exploits["cve_id"])
]

print(f"Moduli Metasploit: {len(msf_data)}")
print(f"CVE uniche: {df_msf['cve_id'].nunique()}")
print(f"CVE associate a exploit: {df_msf_exploits['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(msf_matches)}")
print(f"Possibili CVE da recuperare: {len(msf_recovery)}")

display(df_msf.head())


## 7. Exploit-DB — Metadati

Indicizzazione delle evidenze pubbliche tramite CVE ID.

In [ ]:
# Cella 7: Acquisizione dati Exploit-DB

EDB_URL = (
    "https://gitlab.com/exploit-database/"
    "exploitdb/-/raw/main/files_exploits.csv"
)

edb_file = RAW_DIR / "exploitdb.csv"

# Download con caching locale
if not edb_file.exists():
    response = requests.get(EDB_URL, timeout=120)
    response.raise_for_status()
    edb_file.write_bytes(response.content)

# Lettura del dataset
edb_data = pd.read_csv(edb_file).fillna("")

edb_records = []

for _, row in edb_data.iterrows():

    cve_ids = set(re.findall(
        r"CVE-\d{4}-\d{4,}",
        str(row.get("codes", "")).upper()
    ))

    for cve_id in sorted(cve_ids):

        edb_records.append({
            "cve_id": cve_id,
            "edb_id": row.get("id"),
            "edb_description": row.get("description"),
            "edb_type": row.get("type"),
            "edb_platform": row.get("platform"),
            "edb_verified": row.get("verified"),
            "edb_date": row.get("date_published")
        })

df_edb = pd.DataFrame(edb_records).drop_duplicates()

df_edb.to_csv(
    PROCESSED_DIR / "exploitdb.csv", index=False
)

# Incrocio con le CVE candidate
edb_matches = df_candidates[
    df_candidates["cve_id"].isin(df_edb["cve_id"])
]

# Possibili CVE da recuperare
edb_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_edb["cve_id"])
]

print(f"Exploit indicizzati: {len(edb_data)}")
print(f"CVE uniche: {df_edb['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(edb_matches)}")
print(f"Possibili CVE da recuperare: {len(edb_recovery)}")

display(df_edb.head())


## 8. GitHub — Riferimenti NVD

Classificazione dei riferimenti già presenti in NVD; non si scaricano i contenuti dei repository. Un riferimento non è necessariamente un PoC.

In [ ]:
# Cella 8: Acquisizione riferimenti GitHub

from urllib.parse import urlparse

github_records = []

df_search = pd.concat([df_av, df_kernel], ignore_index=True)[["cve_id", "references"]].drop_duplicates("cve_id")

for row in df_search.itertuples(index=False):

    references = json.loads(row.references)

    for url in references:

        try:
            parsed = urlparse(url)
            hostname = parsed.hostname
        except ValueError:
            continue

        if hostname not in ["github.com", "www.github.com"]:
            continue

        parts = parsed.path.strip("/").split("/")

        if len(parts) < 2:
            continue

        repository = None if parts[0] == "advisories" else "/".join(parts[:2])
        reference_type = "other"

        if "advisories" in parts:
            reference_type = "advisory"

        elif "commit" in parts:
            reference_type = "commit"

        elif "issues" in parts:
            reference_type = "issue"

        elif "pull" in parts:
            reference_type = "pull_request"

        elif len(parts) == 2:
            reference_type = "repository"

        elif "blob" in parts:
            reference_type = "file"

        github_records.append({
            "cve_id": row.cve_id,
            "github_repository": repository,
            "github_url": url,
            "github_type": reference_type
        })

df_github = pd.DataFrame(
    github_records,
    columns=[
        "cve_id", "github_repository",
        "github_url", "github_type"
    ]
).drop_duplicates()

df_github.to_csv(
    PROCESSED_DIR / "github_references.csv", index=False
)

github_matches = df_candidates[
    df_candidates["cve_id"].isin(df_github["cve_id"])
]

github_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_github["cve_id"])
]

print(f"Riferimenti GitHub: {len(df_github)}")
print(f"CVE uniche: {df_github['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(github_matches)}")
print(f"Corrispondenze con piattaforma incerta: {len(github_recovery)}")

print("\nDistribuzione dei riferimenti:")
print(df_github["github_type"].value_counts())

display(df_github.head())


## 9. Falco — Riferimenti pubblici

Associazioni esplicite nelle regole YAML. Non rileva regole generiche, commenti o tracce di attacco; nessun test del sensore.

In [ ]:
# Cella 9: Acquisizione regole Falco

FALCO_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "falcosecurity/rules/main/rules/"
)

FALCO_FILES = {
    "stable": "falco_rules.yaml",
    "incubating": "falco-incubating_rules.yaml",
    "sandbox": "falco-sandbox_rules.yaml"
}

falco_records = []
total_rules = 0

for maturity, filename in FALCO_FILES.items():

    file_path = RAW_DIR / filename

    # Download con caching locale
    if not file_path.exists():
        response = requests.get(
            FALCO_BASE_URL + filename, timeout=60
        )
        response.raise_for_status()
        file_path.write_bytes(response.content)

    # Lettura del file YAML
    rules = yaml.safe_load(
        file_path.read_text(encoding="utf-8")
    )

    for rule in rules:

        if not isinstance(rule, dict) or "rule" not in rule:
            continue

        total_rules += 1

        cve_ids = set(re.findall(
            r"CVE-\d{4}-\d{4,}",
            json.dumps(rule).upper()
        ))

        for cve_id in sorted(cve_ids):

            falco_records.append({
                "cve_id": cve_id,
                "falco_rule": rule.get("rule"),
                "falco_description": rule.get("desc"),
                "falco_priority": rule.get("priority"),
                "falco_maturity": maturity,
                "falco_source": rule.get("source", "syscall"),
                "falco_url": (
                    "https://github.com/falcosecurity/"
                    "rules/blob/main/rules/" + filename
                )
            })

df_falco = pd.DataFrame(
    falco_records,
    columns=[
        "cve_id", "falco_rule", "falco_description",
        "falco_priority", "falco_maturity",
        "falco_source", "falco_url"
    ]
).drop_duplicates()

df_falco.to_csv(
    PROCESSED_DIR / "falco_rules.csv", index=False
)

falco_matches = df_candidates[
    df_candidates["cve_id"].isin(df_falco["cve_id"])
]

falco_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_falco["cve_id"])
]

print(f"Regole Falco analizzate: {total_rules}")
print(f"Associazioni CVE-regola: {len(df_falco)}")
print(f"CVE uniche: {df_falco['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(falco_matches)}")
print(f"Possibili CVE da recuperare: {len(falco_recovery)}")

display(df_falco.head())


## 10. Incrocio, recupero e pertinenza host

Una riga per CVE. Recupero provvisorio con ambiente Vulhub o indicazione Linux esplicita nelle fonti; le incompatibilità CPE sono accantonate, non sovrascritte automaticamente. Le parole chiave descrivono possibili effetti osservabili, non la detection effettiva.

In [ ]:
# Classificazione conservativa della piattaforma dei metadati.
PORTABLE_PLATFORMS = {"multi", "multiple", "unix", "posix", "webapps", "webapp", "php", "java", "jsp", "python", "ruby", "perl", "cgi", "nodejs", "node", "javascript"}
OTHER_PLATFORMS = {"windows", "win", "win32", "win64", "osx", "macos", "android", "ios", "freebsd", "openbsd", "netbsd", "solaris", "aix", "netware"}
def platform_fit(value):
    tokens = set(re.findall(r"[a-z0-9]+", str(value).lower()))
    if "linux" in tokens:
        return "linux"
    if tokens & OTHER_PLATFORMS:
        return "other_or_mixed"
    if tokens & PORTABLE_PLATFORMS:
        return "portable"
    return "unknown"

msf_platform = df_msf_exploits["msf_platform"].map(platform_fit)
edb_platform = df_edb["edb_platform"].map(platform_fit)
msf_linux_ids = set(df_msf_exploits.loc[msf_platform.eq("linux"), "cve_id"])
edb_linux_ids = set(df_edb.loc[edb_platform.eq("linux"), "cve_id"])
msf_compatible_ids = set(df_msf_exploits.loc[msf_platform.isin(["linux", "portable"]), "cve_id"])
edb_compatible_ids = set(df_edb.loc[edb_platform.isin(["linux", "portable"]), "cve_id"])
vulhub_ids = set(df_vulhub["cve_id"])
falco_syscall_ids = set(df_falco.loc[df_falco["falco_source"].eq("syscall"), "cve_id"])

# Falco e GitHub da soli non attestano la piattaforma della CVE.
recovery_ids = vulhub_ids | msf_linux_ids | edb_linux_ids
# Non recuperiamo automaticamente i casi con CPE esclusivamente non Linux.
df_recovered = df_unknown[df_unknown["cve_id"].isin(recovery_ids)].copy()
df_conflicts = df_nonlinux[df_nonlinux["cve_id"].isin(recovery_ids | falco_syscall_ids)].copy()
df_pending = df_unknown[~df_unknown["cve_id"].isin(recovery_ids)].copy()
df_initial = df_candidates.copy()
df_initial["platform_basis"] = df_initial["platform_nvd"]
df_recovered["platform_basis"] = "external_provisional"

df_enriched = pd.concat([df_initial, df_recovered], ignore_index=True)
assert df_enriched["cve_id"].is_unique and not df_enriched["kernel"].any()

sources = {"vulhub": df_vulhub, "metasploit": df_msf_exploits, "exploitdb": df_edb,
           "github": df_github, "falco": df_falco}
# Le colonne has_* sono match grezzi; le colonne compatible indicano solo compatibilità documentale potenziale.
def enrich(frame):
    frame = frame.copy()
    for name, source in sources.items():
        counts = source.groupby("cve_id").size()
        frame[f"{name}_count"] = frame["cve_id"].map(counts).fillna(0).astype(int)
        frame[f"has_{name}"] = frame[f"{name}_count"].gt(0)
    for name, ids in {"msf_linux": msf_linux_ids, "edb_linux": edb_linux_ids,
                      "msf_compatible": msf_compatible_ids, "edb_compatible": edb_compatible_ids,
                      "falco_syscall": falco_syscall_ids}.items():
        frame[name] = frame["cve_id"].isin(ids)
    frame["source_count"] = frame[[f"has_{name}" for name in sources]].sum(axis=1)
    evidence_cols = ["has_vulhub", "msf_compatible", "edb_compatible", "has_github", "falco_syscall"]
    frame["context_source_count"] = frame[evidence_cols].sum(axis=1)
    return frame

df_enriched = enrich(df_enriched)
df_kernel_enriched = enrich(df_kernel)  # Separato; non entra nel ranking principale.

# Indicatori testuali semplici, ricavati dalla descrizione NVD e dai tag Vulhub.
# Non filtrano le CVE e non sostituiscono una traccia syscall.
HOST_PATTERNS = {
    "process_execution": r"\b(?:remote code execution|arbitrary code execution|command execution|command injection|execute (?:arbitrary |system |shell )?(?:code|commands)|execution of arbitrary (?:code|commands)|rce)\b",
    "privilege_change": r"\b(?:privilege escalation|privilege elevation|elevation of privilege|escalat(?:e|ion of) privileges|gain (?:root|elevated|administrative) privileges|arbitrary privilege escalation)\b",
    "file_write": r"\b(?:arbitrary file writ(?:e|ing)|file overwrite|write arbitrary files?|overwrite (?:arbitrary |system )?files?|arbitrary file upload|file upload)\b",
    "file_read": r"\b(?:arbitrary file read|read arbitrary files?|local file inclusion|file disclosure|read sensitive files?)\b",
    "container_escape": r"\b(?:container escape|escape (?:from )?(?:the |a )?container|sandbox escape)\b",
}
vulhub_tags = df_vulhub.groupby("cve_id")["vulhub_tags"].agg(lambda s: " ".join(sorted(set(s.dropna().astype(str)))))
text = df_enriched["description"].fillna("") + " " + df_enriched["cve_id"].map(vulhub_tags).fillna("")
compiled = {key: re.compile(pattern, re.I) for key, pattern in HOST_PATTERNS.items()}
df_enriched["host_signals"] = text.map(lambda s: ";".join(key for key, pattern in compiled.items() if pattern.search(s)))
df_enriched["host_hint"] = df_enriched["host_signals"].ne("")
# Conserviamo l'origine dei match: quelli compatibili sono separati dai riferimenti grezzi.
# La compatibilità è documentale e le piattaforme portabili restano da verificare.
evidence_tables = [
    ("vulhub_paths", df_vulhub, "vulhub_path"),
    ("msf_modules", df_msf_exploits, "msf_module"),
    ("msf_compatible_modules", df_msf_exploits.loc[msf_platform.isin(["linux", "portable"])], "msf_module"),
    ("edb_ids", df_edb, "edb_id"),
    ("edb_compatible_ids", df_edb.loc[edb_platform.isin(["linux", "portable"])], "edb_id"),
    ("falco_rules", df_falco, "falco_rule"),
    ("github_reference_types", df_github, "github_type"),
]
for col, source, field in evidence_tables:
    values = source.groupby("cve_id")[field].agg(lambda s: json.dumps(sorted(set(s.dropna().astype(str))), ensure_ascii=False))
    for frame in (df_enriched, df_kernel_enriched):
        frame[col] = frame["cve_id"].map(values).fillna("[]")

for name, frame in {"candidates_enriched": df_enriched, "kernel_enriched": df_kernel_enriched,
                    "platform_pending": df_pending, "platform_conflicts": df_conflicts}.items():
    frame.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)
print(f"Candidate NVD dirette/contestuali: {len(df_initial)}")
print(f"Recuperi provvisori: {len(df_recovered)}")
print(f"Conflitti da rivedere: {len(df_conflicts)}")
print(f"Piattaforma ancora incerta: {len(df_pending)}")
print(f"Candidate dopo integrazione: {len(df_enriched)}")
print(f"Kernel arricchito e separato: {len(df_kernel_enriched)}")
print(f"Candidate con potenziale segnale host testuale: {df_enriched['host_hint'].sum()}")
print("\nCopertura grezza delle fonti:")
print(df_enriched[[f"has_{name}" for name in sources]].sum())


## 11. Ranking preliminare — scala 0–100

Somma di cinque indicatori binari: una fonte contribuisce al massimo una volta, anche se contiene più riferimenti. I pesi esprimono una preferenza progettuale ispirata alla raccolta multi-fonte e alla riproducibilità (CVEfixes, Jacobs et al., Mu et al., CVE-GENIE), non un algoritmo o una scala validati da tali lavori.

| Evidenza documentale | Peso iniziale | Motivazione per CVE2Patch |
|---|---:|---|
| Vulhub | 30 | Ambiente documentato: utile per predisporre un laboratorio. |
| Metasploit compatibile | 25 | Modulo classificato exploit con piattaforma Linux o potenzialmente portabile. |
| Exploit-DB compatibile | 15 | Evidenza pubblica complementare, da verificare. |
| Falco (syscall) | 25 | Regola pubblica con riferimento esplicito al CVE ID: rilevanza per il progetto, non detection dimostrata. |
| GitHub | 5 | Riferimento NVD a GitHub: contesto aggiuntivo, non necessariamente PoC. |
| **Totale** | **100** | Punteggio documentale massimo. |

A parità di punteggio: AV Local, poi indizio testuale host, poi CVE ID. Nessun punteggio per l'indizio testuale; nessun filtro su un singolo tipo di evidenza. I riferimenti possono descrivere lo stesso artefatto; la somma non misura indipendenza o quantità di prove. Le candidate a punteggio zero restano nel ranking, ma non nel pool di discussione. Nessun log pubblico strutturato è stato integrato. Le regole Falco esistenti vanno tenute distinte dalla futura generazione e valutazione (rischio di leakage).


In [ ]:
df_ranked = df_enriched.copy()

# Una sola somma pesata; nessun bonus aggiuntivo.
assert sum(WEIGHTS.values()) == 100
assert df_ranked[list(WEIGHTS)].notna().all().all()
df_ranked["score"] = sum(df_ranked[col].astype(int) * weight for col, weight in WEIGHTS.items())
assert df_ranked["score"].between(0, 100).all()

df_ranked["local_priority"] = df_ranked["attack_vector"].eq("LOCAL").astype(int)
df_ranked = df_ranked.sort_values(["score", "local_priority", "host_hint", "cve_id"],
                                  ascending=[False, False, False, True]).reset_index(drop=True)
df_ranked["ranking"] = df_ranked.index + 1

# Pool esplorativo: nessuna quota e nessuna fonte obbligatoria.
df_pool = df_ranked[df_ranked["score"].gt(0)].head(POOL_SIZE).copy()
assert not df_ranked["kernel"].any() and df_ranked["cve_id"].is_unique
assert len(df_pool) <= POOL_SIZE
for filename, frame in [("ranked_cves.csv", df_ranked), ("initial_pool.csv", df_pool)]:
    frame.to_csv(OUTPUT_DIR / filename, index=False)

view_cols = ["ranking", "cve_id", "attack_vector", "platform_basis", "has_vulhub", "msf_compatible",
             "edb_compatible", "has_github", "falco_syscall", "host_signals", "score"]
print(f"CVE ordinate: {len(df_ranked)}")
print(f"CVE con contesto aggiuntivo: {df_ranked['score'].gt(0).sum()}")
print(f"CVE nel pool: {len(df_pool)} (richieste: {POOL_SIZE})")
print(f"Massimo teorico: {sum(WEIGHTS.values())}; massimo osservato: {df_ranked['score'].max()}")
print(df_ranked["score"].value_counts().sort_index())
display(df_pool[view_cols])


## 12. Risultati e revisione manuale

Confronto Network/Local, pool alternativo Local e piccolo campione per il confronto con il professore. Manifest con configurazione e hash dei file originali (non si assume che la data di esecuzione coincida con la data di download).

In [ ]:
print("Distribuzione AV delle candidate:")
print(df_ranked["attack_vector"].value_counts())
print("\nDistribuzione AV del pool:")
print(df_pool["attack_vector"].value_counts())
summary = df_ranked.groupby("attack_vector")["score"].agg(["count", "mean", "median", "max"])
display(summary.round(2))
print("\nProvenienza delle evidenze di piattaforma nel pool:")
print(df_pool["platform_basis"].value_counts())
print("\nIndizi host nel pool (non verificati):")
print(df_pool["host_hint"].value_counts())

df_local_pool = df_ranked[df_ranked["attack_vector"].eq("LOCAL") & df_ranked["score"].gt(0)].head(POOL_SIZE).copy()
df_local_pool.to_csv(OUTPUT_DIR / "local_pool_preview.csv", index=False)
print("\nPool alternativo Local:")
display(df_local_pool[view_cols].head(20))

# Campione di revisione: non è un set congelato.
df_review = pd.concat([df_pool.head(10), df_local_pool.head(10)]).drop_duplicates("cve_id").copy()
df_review["review_platform"] = "TODO"
df_review["review_host_relevance"] = "TODO"
df_review["review_context"] = "TODO"
df_review["review_notes"] = ""
df_review.to_csv(OUTPUT_DIR / "manual_review.csv", index=False)

def file_hash(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

input_files = list(NVD_FILES) + [vulhub_file, msf_file, edb_file] + [RAW_DIR / name for name in FALCO_FILES.values()]
manifest = {
    "method_version": "3.0",
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "score_max_theoretical": sum(WEIGHTS.values()),
    "ranking_tiebreak": ["local_priority desc", "host_hint desc", "cve_id asc"],
    "literature": ["10.1145/3475960.3475985", "10.1016/j.jss.2023.111679", "10.1093/cybsec/tyaa015", "https://www.usenix.org/conference/usenixsecurity18/presentation/mu", "arXiv:2509.01835v2"],
    "start_date": START_DATE, "end_date": END_DATE,
    "attack_vectors": ATTACK_VECTORS, "kernel_separate": True,
    "pool_size_requested": POOL_SIZE, "weights": WEIGHTS,
    "counts": {"nvd_unique": len(df_all), "valid_in_period": len(df_valid), "kernel": len(df_kernel),
               "av_non_kernel": len(df_av), "nvd_linux": len(df_initial), "recovered": len(df_recovered),
               "pending": len(df_pending), "conflicts": len(df_conflicts), "enriched": len(df_enriched),
               "pool": len(df_pool), "local_preview": len(df_local_pool),
               "positive_score": int(df_ranked["score"].gt(0).sum()),
               "pool_local": int(df_pool["attack_vector"].eq("LOCAL").sum()),
               "pool_external_provisional": int(df_pool["platform_basis"].eq("external_provisional").sum())},
    "inputs": [{"file": str(p.relative_to(BASE_DIR)).replace("\\", "/"), "bytes": p.stat().st_size,
                "sha256": file_hash(p)} for p in input_files],
    "limitations": ["NVD feed annuali per anno ID", "CPE e kernel euristici", "Recupero provvisorio", "Match delle fonti non verificati", "Nessun log o test di Falco", "Pesi empirici non validati dalla letteratura", "Match tra fonti non necessariamente indipendenti", "Host hint soltanto testuale"]
}
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nOutput disponibili in: {OUTPUT_DIR}")
